In [1]:
import numpy as np 
import pandas as pd 
import os

In [2]:
base_path = '/kaggle/input/datasets/zalcode/bubble-answer-dataset/train'

data = []

for img_file in os.listdir(base_path):
    img_path = os.path.join(base_path, img_file)
    if img_file.endswith(('.jpg','.png','.jpeg')):
        data.append([img_path])


df = pd.DataFrame(data, columns=['image_path'])

In [3]:
import pandas as pd
import os
import shutil

# Load excel
df = pd.read_csv('/kaggle/input/datasets/zalcode/bubble-answer-dataset/train/_classes.csv')  # change name if needed

df.columns = df.columns.str.strip()

image_folder = '/kaggle/input/datasets/zalcode/bubble-answer-dataset/train'

# Output (write here)
output_folder = '/kaggle/working/train'

classes = ["crossed", "default", "filled", "invalid"]

# Create folders
for cls in classes:
    os.makedirs(os.path.join(output_folder, cls), exist_ok=True)

# Move images
for _, row in df.iterrows():
    img_name = row['filename']  # adjust column name

    for cls in classes:
        if row[cls] == 1:
            src = os.path.join(image_folder, img_name)
            dst = os.path.join(output_folder, cls, img_name)
            shutil.copy(src, dst)

In [4]:
import os

for root, dirs, files in os.walk('/kaggle/working'):
    print(root, len(files))

/kaggle/working 1
/kaggle/working/train 0
/kaggle/working/train/default 129
/kaggle/working/train/filled 120
/kaggle/working/train/crossed 99
/kaggle/working/train/invalid 144


In [5]:
import pandas as pd
import os
import shutil

df_valid = pd.read_csv('/kaggle/input/datasets/zalcode/bubble-answer-dataset/valid/_classes.csv')
df_valid.columns = df_valid.columns.str.strip()

image_folder_valid = '/kaggle/input/datasets/zalcode/bubble-answer-dataset/valid'
output_folder_valid = '/kaggle/working/valid'

classes = ["crossed", "default", "filled", "invalid"]

for cls in classes:
    os.makedirs(os.path.join(output_folder_valid, cls), exist_ok=True)

for _, row in df_valid.iterrows():
    img_name = row['filename']
    for cls in classes:
        if row[cls] == 1:
            src = os.path.join(image_folder_valid, img_name)
            dst = os.path.join(output_folder_valid, cls, img_name)
            shutil.copy(src, dst)

# Verify
for root, dirs, files in os.walk('/kaggle/working'):
    print(root, len(files))

/kaggle/working 1
/kaggle/working/valid 0
/kaggle/working/valid/default 39
/kaggle/working/valid/filled 31
/kaggle/working/valid/crossed 38
/kaggle/working/valid/invalid 21
/kaggle/working/train 0
/kaggle/working/train/default 129
/kaggle/working/train/filled 120
/kaggle/working/train/crossed 99
/kaggle/working/train/invalid 144


In [6]:
# Uninstall current torch and install CUDA 11.6 compatible version
!pip install torch==1.13.1+cu116 torchvision==0.14.1+cu116 \
    --extra-index-url https://download.pytorch.org/whl/cu116 -q

ERROR: Could not find a version that satisfies the requirement torch==1.13.1+cu116 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0)
ERROR: No matching distribution found for torch==1.13.1+cu116


In [7]:
import torch
print(torch.cuda.get_device_name(0))  # Should say "Tesla T4"
print(torch.cuda.is_available())       # Should say True

Tesla T4
True


In [8]:
import pandas as pd
import os
import shutil

df_test = pd.read_csv('/kaggle/input/datasets/zalcode/bubble-answer-dataset/test/_classes.csv')
df_test.columns = df_test.columns.str.strip()

image_folder_test = '/kaggle/input/datasets/zalcode/bubble-answer-dataset/test'
output_folder_test = '/kaggle/working/test'

classes = ["crossed", "default", "filled", "invalid"]

for cls in classes:
    os.makedirs(os.path.join(output_folder_test, cls), exist_ok=True)

for _, row in df_test.iterrows():
    img_name = row['filename']
    for cls in classes:
        if row[cls] == 1:
            src = os.path.join(image_folder_test, img_name)
            dst = os.path.join(output_folder_test, cls, img_name)
            shutil.copy(src, dst)

# Verify counts
for root, dirs, files in os.walk('/kaggle/working/test'):
    if files:
        print(root, len(files))

/kaggle/working/test/default 7
/kaggle/working/test/filled 9
/kaggle/working/test/crossed 16
/kaggle/working/test/invalid 7


In [9]:
!pip install torchmetrics clean-fid -q

In [10]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import matplotlib.pyplot as plt

# Config
IMAGE_SIZE = 64  # GAN works better at 64x64
LATENT_DIM = 100
BATCH_SIZE  = 32
EPOCHS      = 300
LR          = 0.0002
BETA1       = 0.5
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = ["crossed", "default", "filled", "invalid"]
TRAIN_DIR   = '/kaggle/working/train'
GAN_OUT_DIR = '/kaggle/working/gan_generated'
AUG_OUT_DIR = '/kaggle/working/aug_generated'
FINAL_DIR   = '/kaggle/working/final_dataset'

print(f"Device: {DEVICE}")

# -----------------------------
# Generator
# -----------------------------
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            # input: latent_dim x 1 x 1
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            # 512 x 4 x 4
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # 256 x 8 x 8
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # 128 x 16 x 16
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # 64 x 32 x 32
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
            # 3 x 64 x 64
        )

    def forward(self, z):
        return self.net(z)

# -----------------------------
# Discriminator
# -----------------------------
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # 3 x 64 x 64
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 64 x 32 x 32
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # 128 x 16 x 16
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # 256 x 8 x 8
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            # 512 x 4 x 4
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1)

# Weight initialization
def weights_init(m):
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)
for class_name in CLASS_NAMES:
    train_dcgan(class_name)

Device: cuda


NameError: name 'train_dcgan' is not defined

In [ ]:
# -----------------------------
# Single class dataset
# -----------------------------
class SingleClassDataset(Dataset):
    def __init__(self, class_dir, transform=None):
        self.paths = [
            os.path.join(class_dir, f)
            for f in os.listdir(class_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as img:
            img = img.convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

gan_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5]),
])

# -----------------------------
# Train DCGAN for one class
# -----------------------------
def train_dcgan(class_name, num_epochs=EPOCHS):
    print(f"\n{'='*50}")
    print(f"Training DCGAN for class: {class_name}")
    print(f"{'='*50}")

    class_dir = os.path.join(TRAIN_DIR, class_name)
    dataset = SingleClassDataset(class_dir, transform=gan_tfms)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                         shuffle=True, num_workers=2)

    print(f"Real images: {len(dataset)}")

    # Init models
    G = Generator(LATENT_DIM).to(DEVICE)
    D = Discriminator().to(DEVICE)
    G.apply(weights_init)
    D.apply(weights_init)

    criterion  = nn.BCELoss()
    opt_G = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))

    fixed_noise = torch.randn(16, LATENT_DIM, 1, 1, device=DEVICE)

    G_losses, D_losses = [], []

    for epoch in range(1, num_epochs + 1):
        for real_imgs in loader:
            real_imgs = real_imgs.to(DEVICE)
            b = real_imgs.size(0)

            real_labels = torch.ones(b, device=DEVICE) * 0.9   # label smoothing
            fake_labels = torch.zeros(b, device=DEVICE)

            # ── Train Discriminator ──
            opt_D.zero_grad()
            out_real = D(real_imgs)
            loss_D_real = criterion(out_real, real_labels)

            noise = torch.randn(b, LATENT_DIM, 1, 1, device=DEVICE)
            fake_imgs = G(noise)
            out_fake = D(fake_imgs.detach())
            loss_D_fake = criterion(out_fake, fake_labels)

            loss_D = loss_D_real + loss_D_fake
            loss_D.backward()
            opt_D.step()

            # ── Train Generator ──
            opt_G.zero_grad()
            out_fake2 = D(fake_imgs)
            loss_G = criterion(out_fake2,
                               torch.ones(b, device=DEVICE))
            loss_G.backward()
            opt_G.step()

        G_losses.append(loss_G.item())
        D_losses.append(loss_D.item())

        if epoch % 50 == 0:
            print(f"Epoch {epoch}/{num_epochs} | "
                  f"D Loss: {loss_D.item():.4f} | "
                  f"G Loss: {loss_G.item():.4f}")

    # Plot losses
    plt.figure(figsize=(8, 4))
    plt.plot(G_losses, label='Generator')
    plt.plot(D_losses, label='Discriminator')
    plt.title(f'GAN Loss - {class_name}')
    plt.xlabel('Epoch')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'gan_loss_{class_name}.png', dpi=150)
    plt.show()

    return G

In [ ]:
import cv2

# -----------------------------
# Blur detection filter
# -----------------------------
def is_sharp(img_path, threshold=50.0):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return False
    laplacian_var = cv2.Laplacian(img, cv2.CV_64F).var()
    return laplacian_var > threshold

# -----------------------------
# Generate images from trained G
# -----------------------------
def generate_gan_images(G, class_name, num_generate=1200):
    out_dir = os.path.join(GAN_OUT_DIR, class_name)
    os.makedirs(out_dir, exist_ok=True)

    G.eval()
    saved = 0
    batch = 64

    with torch.no_grad():
        while saved < num_generate:
            noise = torch.randn(batch, LATENT_DIM, 1, 1, device=DEVICE)
            fake  = G(noise)
            # denormalize: [-1,1] → [0,1]
            fake  = (fake + 1) / 2.0

            for i in range(fake.size(0)):
                if saved >= num_generate:
                    break
                path = os.path.join(out_dir, f"gan_{saved:05d}.png")
                save_image(fake[i], path)
                saved += 1

    print(f"Generated {saved} raw GAN images for '{class_name}'")

    # Filter blurry images
    all_files = [os.path.join(out_dir, f)
                 for f in os.listdir(out_dir)]
    sharp_files = [f for f in all_files if is_sharp(f)]
    blurry      = len(all_files) - len(sharp_files)

    print(f"Sharp: {len(sharp_files)} | Blurry removed: {blurry}")
    return sharp_files

In [ ]:
from torchvision import transforms as T
import random

aug_tfms = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.3),
    T.RandomRotation(30),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3),
    T.RandomPerspective(distortion_scale=0.3, p=0.5),
    T.GaussianBlur(kernel_size=3),
    T.RandomGrayscale(p=0.1),
])

def augment_real_images(class_name, num_needed):
    real_dir = os.path.join(TRAIN_DIR, class_name)
    out_dir  = os.path.join(AUG_OUT_DIR, class_name)
    os.makedirs(out_dir, exist_ok=True)

    real_images = [
        f for f in os.listdir(real_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    saved = 0
    while saved < num_needed:
        fname = random.choice(real_images)
        with Image.open(os.path.join(real_dir, fname)) as img:
            img = img.convert('RGB')
        aug = aug_tfms(img)
        aug.save(os.path.join(out_dir, f"aug_{saved:05d}.png"))
        saved += 1

    print(f"Augmented {saved} images for '{class_name}'")

In [ ]:
TARGET = 1500

def build_final_dataset(class_name, gan_sharp_files):
    final_dir = os.path.join(FINAL_DIR, class_name)
    os.makedirs(final_dir, exist_ok=True)

    count = 0

    # 1 - Copy real images
    real_dir = os.path.join(TRAIN_DIR, class_name)
    for f in os.listdir(real_dir):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            src = os.path.join(real_dir, f)
            dst = os.path.join(final_dir, f"real_{f}")
            shutil.copy(src, dst)
            count += 1

    # 2 - Copy GAN images (up to 1000)
    gan_limit = min(1000, len(gan_sharp_files), TARGET - count)
    for i, f in enumerate(gan_sharp_files[:gan_limit]):
        dst = os.path.join(final_dir, f"gan_{i:05d}.png")
        shutil.copy(f, dst)
        count += 1

    # 3 - Augment to fill remaining gap
    remaining = TARGET - count
    if remaining > 0:
        print(f"Filling {remaining} with augmentation...")
        augment_real_images(class_name, remaining)
        aug_dir = os.path.join(AUG_OUT_DIR, class_name)
        for f in os.listdir(aug_dir):
            src = os.path.join(aug_dir, f)
            dst = os.path.join(final_dir, f"aug_{f}")
            shutil.copy(src, dst)
            count += 1

    print(f"✅ {class_name}: {count} total images in final dataset")

In [ ]:
import shutil

all_generators = {}

# Train GAN + generate + build final for each class
for cls in CLASS_NAMES:
    # Train DCGAN
    G = train_dcgan(cls, num_epochs=EPOCHS)
    all_generators[cls] = G

    # Generate & filter
    sharp_files = generate_gan_images(G, cls, num_generate=1200)

    # Build final dataset
    build_final_dataset(cls, sharp_files)

# Verify final counts
print("\n📊 Final Dataset Summary:")
for cls in CLASS_NAMES:
    path = os.path.join(FINAL_DIR, cls)
    count = len(os.listdir(path))
    print(f"  {cls}: {count} images")

In [ ]:
for root, dirs, files in os.walk('/kaggle/working'):
    print(root, len(files))

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Config
# -----------------------------
CLASS_NAMES = ["crossed", "default", "filled", "invalid"]
num_classes  = len(CLASS_NAMES)
name2id      = {c: i for i, c in enumerate(CLASS_NAMES)}

FINAL_DIR  = '/kaggle/working/final_dataset'   # 6000 GAN images
SPLIT_DIR  = '/kaggle/working/split_dataset'   # 70/15/15 split

BATCH_SIZE = 32
EPOCHS     = 40
LR         = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# Step 1 — 70 / 15 / 15 Split
# -----------------------------
for split in ['train', 'valid', 'test']:
    for cls in CLASS_NAMES:
        os.makedirs(os.path.join(SPLIT_DIR, split, cls), exist_ok=True)

print("\n📂 Splitting dataset 70/15/15...")
for cls in CLASS_NAMES:
    cls_dir   = os.path.join(FINAL_DIR, cls)
    all_files = [
        f for f in os.listdir(cls_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    # 70% train / 30% temp
    train_files, temp_files = train_test_split(
        all_files, test_size=0.30, random_state=42
    )
    # 50/50 on temp → 15% valid / 15% test
    val_files, test_files = train_test_split(
        temp_files, test_size=0.50, random_state=42
    )

    for f in train_files:
        shutil.copy(os.path.join(cls_dir, f),
                    os.path.join(SPLIT_DIR, 'train', cls, f))
    for f in val_files:
        shutil.copy(os.path.join(cls_dir, f),
                    os.path.join(SPLIT_DIR, 'valid', cls, f))
    for f in test_files:
        shutil.copy(os.path.join(cls_dir, f),
                    os.path.join(SPLIT_DIR, 'test',  cls, f))

    print(f"  {cls}: Train {len(train_files)} | "
          f"Valid {len(val_files)} | Test {len(test_files)}")

# Verify
print("\n📊 Split Summary:")
for split in ['train', 'valid', 'test']:
    total = sum(
        len(os.listdir(os.path.join(SPLIT_DIR, split, cls)))
        for cls in CLASS_NAMES
    )
    print(f"  {split}: {total} images")

# -----------------------------
# Step 2 — Build DataFrames
# -----------------------------
TRAIN_DIR = os.path.join(SPLIT_DIR, 'train')
VAL_DIR   = os.path.join(SPLIT_DIR, 'valid')
TEST_DIR  = os.path.join(SPLIT_DIR, 'test')

def build_df_from_folder(root_dir):
    data = []
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(root_dir, cls)
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                data.append({
                    'image_path': os.path.join(cls_dir, fname),
                    'label':      cls,
                    'label_id':   name2id[cls]
                })
    return pd.DataFrame(data)

train_df = build_df_from_folder(TRAIN_DIR)
val_df   = build_df_from_folder(VAL_DIR)
test_df  = build_df_from_folder(TEST_DIR)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(train_df['label'].value_counts())

# -----------------------------
# Step 3 — Dataset Class
# -----------------------------
class ImageDFDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row['image_path']) as im:
            im = im.convert('RGB')
        if self.transform:
            im = self.transform(im)
        return im, int(row['label_id'])

# -----------------------------
# Step 4 — Transforms
# -----------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_ds = ImageDFDataset(train_df, transform=train_tfms)
val_ds   = ImageDFDataset(val_df,   transform=val_tfms)
test_ds  = ImageDFDataset(test_df,  transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# -----------------------------
# Step 5 — Model: VGG19
# -----------------------------
try:
    from torchvision.models import VGG19_Weights
    model = models.vgg19(weights=VGG19_Weights.IMAGENET1K_V1)
except Exception:
    model = models.vgg19(pretrained=True)

for p in model.features.parameters():
    p.requires_grad = False

in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, num_classes)
model = model.to(device)

# -----------------------------
# Step 6 — Loss, Optimizer, Scheduler
# -----------------------------
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(num_classes),
    y=train_df['label_id'].values
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion  = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer  = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# -----------------------------
# Step 7 — Train / Eval Function
# -----------------------------
def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        total_loss    += loss.item() * images.size(0)
        preds          = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total         += labels.size(0)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    return total_loss / total, total_correct / total, \
           np.array(all_preds), np.array(all_labels)

# -----------------------------
# Step 8 — Training Loop
# -----------------------------
best_val_loss = float('inf')
best_path     = "vgg19_best.pt"

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, _, _           = run_epoch(model, train_loader, train=True)
    val_loss,   val_acc,   y_pred, y_true = run_epoch(model, val_loader,   train=False)

    scheduler.step(val_loss)

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} || "
          f"Val Loss {val_loss:.4f} Acc {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({'model_state': model.state_dict(),
                    'class_names': CLASS_NAMES}, best_path)
        print(f"✅ Saved best model → val loss {best_val_loss:.4f}")

# -----------------------------
# Step 9 — Evaluate on Validation
# -----------------------------
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

val_loss, val_acc, y_pred, y_true = run_epoch(model, val_loader, train=False)
print(f"\nBest Model → Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

print("\nValidation Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

# -----------------------------
# Step 10 — Evaluate on Test
# -----------------------------
test_loss, test_acc, y_pred_test, y_true_test = run_epoch(model, test_loader, train=False)
print(f"\nTest Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

print("\nTest Classification Report:")
print(classification_report(y_true_test, y_pred_test,
                             target_names=CLASS_NAMES, digits=4))

report_dict = classification_report(
    y_true_test, y_pred_test,
    target_names=CLASS_NAMES, digits=4, output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv("vgg19_classification_report.csv", index=True)

# -----------------------------
# Step 11 — Confusion Matrices
# -----------------------------
def plot_confusion_matrix(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES, cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.show()

plot_confusion_matrix(y_true,      y_pred,      
                      "Confusion Matrix - VGG19 (Val)",
                      "vgg19_confusion_matrix_val.png")

plot_confusion_matrix(y_true_test, y_pred_test,
                      "Confusion Matrix - VGG19 (Test)",
                      "vgg19_confusion_matrix_test.png")

# -----------------------------
# Step 12 — Per-class PRF Chart
# -----------------------------
def plot_prf(report_df, title, filename):
    cls_df = report_df.loc[CLASS_NAMES, ['precision', 'recall', 'f1-score']]
    x      = np.arange(len(cls_df))
    width  = 0.26

    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - width, cls_df['precision'], width, label='Precision')
    b2 = ax.bar(x,          cls_df['recall'],   width, label='Recall')
    b3 = ax.bar(x + width,  cls_df['f1-score'], width, label='F1-score')

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
    ax.legend(loc='lower right')

    for bars in (b1, b2, b3):
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f'{h:.2f}',
                        xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 3), textcoords='offset points',
                        ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.show()

plot_prf(report_df,
         'Per-class Precision / Recall / F1 (Test)',
         'vgg19_per_class_prf_test.png')

# -----------------------------
# Step 13 — Parameter Count
# -----------------------------
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 VGG19 Parameters:")
print(f"  Total      : {total_params:,}")
print(f"  Trainable  : {trainable_params:,}")
print(f"  Frozen     : {total_params - trainable_params:,}")

In [ ]:
import h5py
import numpy as np

def save_model_hdf5(model, class_names, filepath='vgg19_model.h5'):
    with h5py.File(filepath, 'w') as f:
        # Save each layer's weights
        for name, param in model.state_dict().items():
            f.create_dataset(name, data=param.cpu().numpy())
        
        # Save class names as metadata
        f.attrs['class_names'] = class_names
        f.attrs['num_classes'] = len(class_names)
        f.attrs['architecture'] = 'VGG19'

    print(f"✅ Model saved to {filepath}")
    size = os.path.getsize(filepath) / (1024 * 1024)
    print(f"   File size: {size:.1f} MB")

save_model_hdf5(model, CLASS_NAMES, 'vgg19_model.h5')

In [ ]:
def load_model_hdf5(filepath, device):
    from torchvision import models
    import torch.nn as nn

    # Rebuild model architecture
    try:
        from torchvision.models import VGG19_Weights
        model = models.vgg19(weights=None)
    except:
        model = models.vgg19(pretrained=False)

    num_classes = 4
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)

    # Load weights from HDF5
    state_dict = {}
    with h5py.File(filepath, 'r') as f:
        for key in f.keys():
            state_dict[key] = torch.tensor(np.array(f[key]))
        class_names = list(f.attrs['class_names'])

    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    print(f" Model loaded from {filepath}")
    print(f"   Classes: {class_names}")
    return model, class_names

# Load
model_loaded, class_names = load_model_hdf5('vgg19_model.h5', device)

In [ ]:
# Save PyTorch format (recommended for resuming training)
torch.save({
    'model_state': model.state_dict(),
    'class_names': CLASS_NAMES
}, 'vgg19_best.pt')

# Save HDF5 format (for sharing / deployment)
save_model_hdf5(model, CLASS_NAMES, 'vgg19_model.h5')

print("Saved in both formats!")

In [ ]:
# 1. PyTorch
torch.save({'model_state': model.state_dict(),
            'class_names': CLASS_NAMES}, 'vgg19_best.pt')

# 2. HDF5
save_model_hdf5(model, CLASS_NAMES, 'vgg19_model.h5')